# 01 — Metro de Santiago Origin–Destination Data Catalog & Table Descriptions (2019 vs. 2023)

This notebook orders, catalogs, and profiles the raw datasets under `data/` (~20.7 GB total) across:
- **April 2019 (`data/2019/`)**: `etapas` (1.31 GB), `viajes` (2.25 GB), and 4 aggregated half-hourly OD / boarding / alighting matrices.
- **April 2023 (`data/2023/`)**: `etapas` (9.41 GB), `viajes` (7.49 GB), and 2 `.xlsb` boarding/alighting matrices.
- **GTFS (`data/GTFS/`)**: 12 transit network feed tables (`stops`, `routes`, `trips`, `stop_times`, `shapes`, etc.).

> **Note:** Every code cell uses `src/metro_od` which works out-of-the-box via streaming samples and scales to full-dataset SQL/DataFrame queries once `duckdb` and `polars` are installed.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from metro_od.catalog import DATASETS, get_dataset_catalog
from metro_od.describe import describe_all_datasets, compare_schemas, format_ascii_table

catalog = get_dataset_catalog()
print(format_ascii_table(catalog, ['key', 'year', 'category', 'format', 'delimiter', 'size_mb', 'quirks_count']))

key                      | year | category  | format   | delimiter | size_mb | quirks_count
-------------------------+------+-----------+----------+-----------+---------+-------------
etapas_2019              | 2019 | etapas    | csv      | ;         | 1246.49 | 3           
viajes_2019              | 2019 | viajes    | csv      | ;         | 2145.12 | 3           
od_comunas_2019          | 2019 | od_matrix | csv      | ;         | 4.73    | 1           
od_zonas_2019            | 2019 | od_matrix | csv      | ;         | 192.11  | 1           
subidas_parada_2019      | 2019 | od_matrix | csv      | ;         | 11.6    | 1           
bajadas_zona_2019        | 2019 | od_matrix | csv      | ;         | 0.09    | 1           
etapas_2023              | 2023 | etapas    | csv      | ;         | 8978.41 | 4           
viajes_2023              | 2023 | viajes    | csv      | ,         | 7141.65 | 5           
matriz_subidas_2023_xlsb | 2023 | od_matrix | xlsb     | N/A       | 6.08    | 1

## 1. Schema Comparison: 2019 vs. 2023 (`etapas` and `viajes`)

Before cleaning and harmonizing 2019 and 2023 into unified Parquet tables, we inspect the structural differences between the two periods.

In [2]:
cmp_etapas = compare_schemas(DATASETS['etapas_2019'], DATASETS['etapas_2023'])
cmp_viajes = compare_schemas(DATASETS['viajes_2019'], DATASETS['viajes_2023'])

print(f"etapas_2019 columns: {cmp_etapas['count_a']} | etapas_2023 columns: {cmp_etapas['count_b']}")
print(f"Shared columns ({len(cmp_etapas['exact_match_columns'])}): {cmp_etapas['exact_match_columns']}")
print(f"Only in 2019: {cmp_etapas['only_in_a']}")
print(f"Added in 2023: {cmp_etapas['only_in_b']}\n")
print(f"viajes_2019 columns: {cmp_viajes['count_a']} | viajes_2023 columns: {cmp_viajes['count_b']}")
print(f"Exact name matches ({len(cmp_viajes['exact_match_columns'])}): {cmp_viajes['exact_match_columns']}")
print(f"Only in 2019 ({len(cmp_viajes['only_in_a'])}): {cmp_viajes['only_in_a'][:15]} ...")
print(f"Only in 2023 ({len(cmp_viajes['only_in_b'])}): {cmp_viajes['only_in_b'][:15]} ...")

etapas_2019 columns: 29 | etapas_2023 columns: 32
Shared columns (29): ['comuna_bajada', 'comuna_subida', 'dist_eucl_paraderos', 'dist_ruta_paraderos', 'f_exp_servicioperiodots', 'fexpansionzonaperiodots', 'mh_bajada', 'mh_subida', 'netapa', 'op', 'par_bajada', 'par_subida', 'servicio_bajada', 'servicio_subida', 'sitio_subida', 'tesperamediaintervalo', 'tiempo2', 'tiempo_bajada', 'tiempo_etapa', 'tiempo_subida', 'tiene_bajada', 'tipo_dia', 'tipo_transporte', 'x_bajada', 'x_subida', 'y_bajada', 'y_subida', 'zona_bajada', 'zona_subida']
Only in 2019: []
Added in 2023: ['periodobajada', 'periodosubida', 'tiempoiniexpedicion']

viajes_2019 columns: 107 | viajes_2023 columns: 101
Exact name matches (10): ['contrato', 'netapassinbajada', 'op_1era_etapa', 'op_2da_etapa', 'op_3era_etapa', 'op_4ta_etapa', 'proposito', 'tipo_corte_etapa_viaje', 'tipodia', 'ultimaetapaconbajada']
Only in 2019 (97): ['linea_metro_subida_4', 'paraderosubida_1era', 'paraderobajada_4ta', 'tiempomediodeviaje', 'parade

## 2. Descriptive Profiles of `etapas` (2019 vs. 2023)

Key anomalies exposed by the table describe:
1. **Mixed Coordinate Systems in 2023 (`x_subida`, `y_subida`)**: In `etapas_2019`, both Bus and Metro use UTM Zone 19S meters (`~350,000`, `~6,290,000`). In `etapas_2023`, Bus uses UTM meters while Metro rows use **WGS84 `(lat, lon)`** (`-33.51`, `-70.75`).
2. **Zone formatting in 2023 (`zona_subida`, `zona_bajada`)**: Stored as float-like strings (`"77.0"`, `"818.0"`) instead of integers (`487`, `476`).
3. **Missing Value Sentinel**: `-` is used extensively across both numeric and categorical fields.

In [7]:
summaries = describe_all_datasets(sample_rows=5_000)
hdrs = ['idx', 'column', 'inferred_type', 'null_or_dash_pct', 'min', 'mean', 'max', 'examples']
for key in ['etapas_2019', 'etapas_2023']:
    s = summaries[key]
    print(f"=== {key} ({s['file_name']}) | {s['size_mb']} MB | {s['num_columns']} cols | {s['rows_sampled']} rows sampled ===")
    print(format_ascii_table(s['columns'], hdrs))
    print()

=== etapas_2019 (etapas20190408_transparencia.csv) | 1246.49 MB | 29 cols | 5000 rows sampled ===
idx | column                  | inferred_type    | null_or_dash_pct | min       | mean         | max       | examples                                                             
----+-------------------------+------------------+------------------+-----------+--------------+-----------+----------------------------------------------------------------------
1   | op                      | numeric          | 0.0              | 1.0       | 4.7532       | 17.0      | ['1', '13', '16']                                                    
2   | netapa                  | numeric          | 0.0              | 1.0       | 1.3212       | 4.0       | ['1', '2', '3']                                                      
3   | tipo_dia                | string/timestamp | 0.0              | None      | None         | None      | ['LABORAL']                                                          
4   | t

## 3. Descriptive Profiles of `viajes` (2019 vs. 2023)

Key anomalies exposed by the `viajes` describe:
1. **Delimiter Change & Trailing Comma**: `viajes_2019` uses `;` (107 columns), whereas `viajes_2023` uses `,` and has a trailing comma on every line creating a 101st empty column (`Unnamed: 100`).
2. **Deprecated Travel Time Column**: In `viajes_2023`, `tviaje` is 100% `-` (deprecated per `Estructura Tabla Viajes 0423.txt`: *"Se anula por nuevo tiempo mejorado"*), replaced by `tviaje2` and stage-level components (`entrada`, `te0`, `tv1`, `tc1`, ..., `egreso`).
3. **Categorical Encoding Shift**: In 2019, `tipodia`, `comunasubida`, `periodosubida`, and `tipotransporte_1era` are strings (`LABORAL`, `LA FLORI`, `BUS`, `METRO`). In 2023, `tipodia`, `comuna_inicio_viaje`, `periodo_inicio_viaje`, and `tipo_transporte_1` are integer codes (`0`, `25`, `63`, `2`).

In [8]:
for key in ['viajes_2019', 'viajes_2023']:
    s = summaries[key]
    print(f"=== {key} ({s['file_name']}) | {s['size_mb']} MB | {s['num_columns']} cols | {s['rows_sampled']} rows sampled ===")
    print(format_ascii_table(s['columns'], hdrs))
    print()

=== viajes_2019 (viajes20190408_transparencia.csv) | 2145.12 MB | 107 cols | 5000 rows sampled ===
idx | column                 | inferred_type    | null_or_dash_pct | min      | mean      | max        | examples                                                                                 
----+------------------------+------------------+------------------+----------+-----------+------------+------------------------------------------------------------------------------------------
1   | netapa                 | numeric          | 0.0              | 1.0      | 1.3934    | 4.0        | ['1', '2', '3']                                                                          
2   | etapas                 | string/timestamp | 100.0            | None     | None      | None       | []                                                                                       
3   | netapassinbajada       | numeric          | 0.0              | 0.0      | 0.2034    | 2.0        | ['0', '1', '2'] 

## 4. Descriptive Profiles of 2019 Aggregated OD / Boarding / Alighting Matrices

Note that `subidas_parada_2019` and `bajadas_zona_2019` use a mixed `|` and `;` delimiter in their data rows (`ParadaSubida|MediaHora;SubidasPromedio`), which our parser automatically splits.

In [5]:
for key in ['od_comunas_2019', 'od_zonas_2019', 'subidas_parada_2019', 'bajadas_zona_2019']:
    s = summaries[key]
    print(f"=== {key} ({s['file_name']}) | {s['size_mb']} MB | {s['num_columns']} cols | {s['rows_sampled']} rows sampled ===")
    print(format_ascii_table(s['columns'], hdrs))
    print()

=== od_comunas_2019 (tablasWeb2019Abril_01_05.MatrizODComunasMediaHora.csv) | 4.73 MB | 13 cols | 5000 rows sampled ===
idx | column               | inferred_type    | null_or_dash_pct | min  | mean    | max       | examples                                        
----+----------------------+------------------+------------------+------+---------+-----------+-------------------------------------------------
1   | ComunaSubida         | string/timestamp | 0.0              | None | None    | None      | ['CERRO NAVIA', 'ESTACION CENTRAL', 'LO PRADO'] 
2   | ComunaBajada         | string/timestamp | 0.0              | None | None    | None      | ['ESTACION CENTRAL', 'LO PRADO', 'MAIPU']       
3   | MediaHora            | string/timestamp | 0.0              | None | None    | None      | ['00:00:00', '00:30:00', '01:00:00']            
4   | ViajeLaboralPromedio | numeric          | 0.0              | 0.2  | 16.0322 | 1580.066  | ['0.6544005871', '1.636001587', '1.963202']     
5   | Viaj

## 5. Full-Table Out-of-Core Describe with DuckDB (Post-Install)

Once `duckdb` is installed (`pip install -r requirements.txt`), run the cell below to execute out-of-core `SUMMARIZE` queries over the full 7–9 GB CSVs and convert them to partitioned Apache Parquet files.

In [6]:
try:
    import duckdb
    con = duckdb.connect()
    for key in ['od_comunas_2019', 'etapas_2019']:
        spec = DATASETS[key]
        print(f'Running DuckDB SUMMARIZE on {key}...')
        df_sum = con.execute(
            f"SUMMARIZE SELECT * FROM read_csv_auto('{spec.path.as_posix()}', delim='{spec.delimiter}', nullstr=['-', ''], ignore_errors=true) LIMIT 100000"
        ).df()
        print(df_sum[['column_name', 'column_type', 'min', 'max', 'approx_unique', 'avg', 'null_percentage']])
except ImportError:
    print('Install DuckDB (`pip install -r requirements.txt`) to run full out-of-core SQL SUMMARIZE.')

Running DuckDB SUMMARIZE on od_comunas_2019...
             column_name column_type          min           max  \
0           ComunaSubida     VARCHAR    CERRILLOS      VITACURA   
1           ComunaBajada     VARCHAR    CERRILLOS      VITACURA   
2              MediaHora        TIME     00:00:00      23:30:00   
3   ViajeLaboralPromedio      DOUBLE  0.200000003   8696.029297   
4           Viajes1Etapa      DOUBLE          0.0   8328.261719   
5          Viajes2Etapas      DOUBLE          0.0   2584.401611   
6          Viajes3Etapas      DOUBLE          0.0   254.5765228   
7          Viajes4Etapas      DOUBLE          0.0   3.527933359   
8      Viajes5oMasEtapas      DOUBLE          0.0  0.8665962219   
9        ViajesUsanMetro      DOUBLE          0.0   5729.080078   
10   ViajesUsanMetroTren      DOUBLE          0.0   139.4777069   
11       ViajesSoloMetro      DOUBLE          0.0   5511.779785   
12         ViajesSoloBus      DOUBLE          0.0   3702.632813   

    approx_uni